In [1]:

# %%
# ==============================================
# Week 4 Section 1 — Hyperparameter Optimization (Business)
# Single-cell; profile-aware; light runtime
# ==============================================

# Config
PROFILE = "dev"   # dev | preprod | final
SECTION_ID = "Wk04_Section1"

CFG = {
    "dev":     dict(max_days=90,   zones=1, test_days=7),
    "preprod": dict(max_days=365,  zones=3, test_days=28),
    "final":   dict(max_days=None, zones=3, test_days=28),
}
cfg = CFG.get(PROFILE, CFG["dev"])

# Paths
from pathlib import Path
BASE_DIR = Path.cwd()
if BASE_DIR.name.lower() == "code":
    BASE_DIR = BASE_DIR.parent
RESULTS_DIR = BASE_DIR / "results" / f"{SECTION_ID}_{PROFILE}"
REPORTS_DIR = RESULTS_DIR / "reports"
PLOTS_DIR   = RESULTS_DIR / "plots"
for d in (REPORTS_DIR, PLOTS_DIR): d.mkdir(parents=True, exist_ok=True)
DATA_PATH   = BASE_DIR / "data" / "Tetuan City power consumption.csv"
CONSOLIDATED = BASE_DIR / "SDS-CP036-powercast_Wk04_Report_Business.md"

print(f"[BOOT] BASE_DIR={BASE_DIR} PROFILE={PROFILE} -> {RESULTS_DIR}")


# Common helpers
from pathlib import Path
import os, re, math, warnings, json
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Cap threads to keep runtime light
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["MALLOC_ARENA_MAX"] = "2"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Optional libraries
_HAVE_SM = False
try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    _HAVE_SM = True
except Exception:
    _HAVE_SM = False

_HAVE_FBPROPHET = False
try:
    try:
        from prophet import Prophet
        _HAVE_FBPROPHET = True
    except Exception:
        from fbprophet import Prophet
        _HAVE_FBPROPHET = True
except Exception:
    _HAVE_FBPROPHET = False

_HAVE_XGB = False
try:
    from xgboost import XGBRegressor as _XGB
    def XGBRegressor(*args, **kwargs):
        kwargs.setdefault("n_estimators", 200)
        kwargs.setdefault("max_depth", 4)
        kwargs.setdefault("learning_rate", 0.1)
        kwargs.setdefault("subsample", 0.9)
        kwargs.setdefault("colsample_bytree", 0.9)
        kwargs.setdefault("tree_method", "hist")
        kwargs.setdefault("n_jobs", 1)
        return _XGB(*args, **kwargs)
    _HAVE_XGB = True
except Exception:
    _HAVE_XGB = False

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [re.sub(r"\s+", " ", c.strip()) for c in df.columns]
    return df

def infer_step_and_freq(idx: pd.DatetimeIndex):
    try:
        freq = pd.infer_freq(idx[:1000])
    except Exception:
        freq = None
    if freq:
        step = pd.to_timedelta(pd.tseries.frequencies.to_offset(freq))
        return step, freq
    diffs = pd.Series(idx).diff().dropna()
    step = diffs.mode().iloc[0] if not diffs.empty else pd.Timedelta(hours=1)
    total_sec = int(step.total_seconds())
    if total_sec % 3600 == 0:
        freq = f"{total_sec // 3600}H"
    elif total_sec % 60 == 0:
        freq = f"{total_sec // 60}T"
    else:
        freq = f"{total_sec}S"
    return step, freq

def limit_last_days(df: pd.DataFrame, date_col="DateTime", max_days=None) -> pd.DataFrame:
    if not max_days:
        return df
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col)
    cutoff = df[date_col].max() - pd.Timedelta(days=max_days)
    return df[df[date_col] >= cutoff]

def pick_zone_cols(df: pd.DataFrame, k: int):
    cols = [c for c in df.columns if re.match(r"Zone\s+\d+\s+Power\s+Consumption", c)]
    cols = sorted(cols, key=lambda s: int(re.search(r"\d+", s).group()))
    return cols[:k] if k else cols

def make_time_splits(df: pd.DataFrame, test_days: int, date_col="DateTime", steps_per_day: int = 24):
    df = df.sort_values(date_col)
    test_len = int(test_days * steps_per_day)
    if test_len >= len(df):
        test_len = max(1, len(df)//10)
    test = df.iloc[-test_len:].copy()
    train_val = df.iloc[:-test_len].copy()
    if len(train_val) > 100:
        cut = int(len(train_val) * 0.85)
        train, val = train_val.iloc[:cut].copy(), train_val.iloc[cut:].copy()
    else:
        train, val = train_val.copy(), pd.DataFrame(columns=train_val.columns)
    return train, val, test

def metrics_series(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred)**2)))
    denom = np.maximum(1e-8, np.where(y_true==0, 1e-8, y_true))
    mape = float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)
    return dict(rmse=rmse, mae=mae, mape=mape)

def save_plot_ts(ts_df: pd.DataFrame, cols, title, path: Path):
    plt.figure(figsize=(10,4))
    for c in cols:
        plt.plot(ts_df.index, ts_df[c], label=c)
    plt.title(title); plt.xlabel("Time"); plt.ylabel("Power")
    plt.legend(loc="best"); plt.tight_layout()
    plt.savefig(path, bbox_inches="tight"); plt.close("all")

def rel(p: Path, base: Path) -> str:
    try: return p.relative_to(base).as_posix()
    except Exception: return p.as_posix()

def write_text(path: Path, text: str):
    path.write_text(text, encoding="utf-8")
    print("Wrote:", rel(path, Path.cwd()))


# Load data
if not DATA_PATH.exists():
    write_text(REPORTS_DIR / "SDS-CP036-powercast_Wk04_Section1_Business_Report.md",
               "# SDS-CP036-powercast – Week 4 Section 1: Hyperparameter Optimization\n\n"
               f"Profile: `{PROFILE}`\n\n**Error:** data file not found: {rel(DATA_PATH, BASE_DIR)}")
    raise SystemExit(0)

df = pd.read_csv(DATA_PATH)
df = normalize_columns(df)
if "DateTime" not in df.columns:
    alts = [c for c in df.columns if c.lower()=="datetime"]
    if alts: df = df.rename(columns={alts[0]:"DateTime"})
zones = pick_zone_cols(df, cfg["zones"])
if not zones:
    raise RuntimeError("No 'Zone X Power Consumption' columns found.")

work = df["DateTime"].to_frame().join(df[zones])
work["DateTime"] = pd.to_datetime(work["DateTime"])
work = work.sort_values("DateTime").set_index("DateTime")

step_td, freq_str = infer_step_and_freq(work.index)
steps_per_hour = max(1, int(round(pd.Timedelta(hours=1) / step_td)))
steps_per_day  = int(24 * steps_per_hour)
print(f"[FREQ] step={step_td} freq='{freq_str}' steps/day={steps_per_day}")

train, val, test = make_time_splits(work.reset_index(), cfg["test_days"], "DateTime", steps_per_day=steps_per_day)
train = train.set_index("DateTime"); val = val.set_index("DateTime"); test = test.set_index("DateTime")

from itertools import product

# ---- Tuning runners ----
def tune_sarimax(series_native, val_index, val_series):
    if not _HAVE_SM: return []
    ser = pd.to_numeric(series_native, errors="coerce").ffill().resample("1H").mean().asfreq("H").ffill()
    if len(ser) > 2400: ser = ser.iloc[-2400:]
    candidates = [((1,0,1),(0,1,1,24)), ((1,1,1),(0,1,1,24))]
    rows = []
    for (p,d,q), (P,D,Q,s) in candidates:
        try:
            model = SARIMAX(ser, order=(p,d,q), seasonal_order=(P,D,Q,s),
                            enforce_stationarity=False, enforce_invertibility=False,
                            simple_differencing=True, trend=None, concentrate_scale=True)
            res = model.fit(method="lbfgs", maxiter=40, disp=False)
            h = int(math.ceil(len(val_index) / steps_per_hour))
            fc = res.get_forecast(steps=h).predicted_mean.values.astype(float)
            idx_hr = pd.date_range(ser.index[-1] + pd.Timedelta(hours=1), periods=h, freq="H")
            native_hours = pd.to_datetime(pd.Index(val_index).floor("H"))
            yhat = pd.Series(fc, index=idx_hr).reindex(native_hours, method="ffill").values[:len(val_index)]
            rows.append({"model":"SARIMAX","params":f"(p,d,q)=({p},{d},{q}),(P,D,Q,s)=({P},{D},{Q},{s})",
                         **metrics_series(val_series, yhat)})
        except Exception as e:
            rows.append({"model":"SARIMAX","params":str(((p,d,q),(P,D,Q,s))),"rmse":np.inf,"mae":np.inf,"mape":np.inf})
    return rows

def make_lag_supervised(y: pd.Series, max_lag=24):
    y = pd.to_numeric(y, errors="coerce").ffill()
    df_sup = pd.DataFrame({"y": y})
    for L in range(1, max_lag+1): df_sup[f"lag_{L}"] = y.shift(L)
    return df_sup.dropna()

def tune_xgb(train_series, val_series, grid):
    if not _HAVE_XGB: return []
    rows = []
    df_tr = make_lag_supervised(train_series, 24)
    X_tr, y_tr = df_tr.drop(columns=["y"]).values, df_tr["y"].values
    for params in grid:
        try:
            model = XGBRegressor(**params)
            model.fit(X_tr, y_tr, verbose=False)
            hist = list(train_series.values)
            preds = []
            for _ in range(len(val_series)):
                feats = [hist[-k] for k in range(1,25)]
                preds.append(float(model.predict(np.array(feats).reshape(1,-1))))
                hist.append(preds[-1])
            m = metrics_series(val_series.values, np.array(preds))
            rows.append({"model":"XGB","params":json.dumps(params), **m})
        except Exception as e:
            rows.append({"model":"XGB","params":json.dumps(params),"rmse":np.inf,"mae":np.inf,"mape":np.inf})
    return rows

# Build grids
xgb_grid = []
if PROFILE=="dev":
    xgb_grid = [{"n_estimators":150,"max_depth":3,"learning_rate":0.1},
                {"n_estimators":250,"max_depth":4,"learning_rate":0.1}]
elif PROFILE=="preprod":
    xgb_grid = [{"n_estimators":n,"max_depth":d,"learning_rate":lr}
                for n in [200,300] for d in [3,4] for lr in [0.05,0.1]]
else:
    xgb_grid = [{"n_estimators":n,"max_depth":d,"learning_rate":lr}
                for n in [200,300,400] for d in [3,4] for lr in [0.05,0.1]]

all_rows = []
best_by_zone = {}
for zone in zones:
    print(f"[ZONE] {zone}")
    val_series = val[zone]
    sarimax_rows = tune_sarimax(pd.concat([train[zone], val[zone]]), val.index, val_series)
    for r in sarimax_rows: r.update({"zone":zone})
    all_rows.extend(sarimax_rows)
    xgb_rows = tune_xgb(train[zone], val[zone], xgb_grid) if len(val) else []
    for r in xgb_rows: r.update({"zone":zone})
    all_rows.extend(xgb_rows)
    cand = [r for r in all_rows if r.get("zone")==zone and np.isfinite(r["rmse"])]
    if cand:
        best = sorted(cand, key=lambda r: r["rmse"])[0]
        best_by_zone[zone] = {"model":best["model"], "params":best["params"], 
                              "rmse":best["rmse"], "mae":best["mae"], "mape":best["mape"]}

tuning_csv = REPORTS_DIR / "tuning_results.csv"
pd.DataFrame(all_rows).to_csv(tuning_csv, index=False)
best_json = REPORTS_DIR / "best_params.json"
best_json.write_text(json.dumps(best_by_zone, indent=2), encoding="utf-8")

print("✅ Saved tuning results to:", rel(tuning_csv, BASE_DIR))
print("✅ Saved best params JSON to:", rel(best_json, BASE_DIR))

# ---- Business report ----
s1_md = REPORTS_DIR / "SDS-CP036-powercast_Wk04_Section1_Business_Report.md"
report = f"""
# SDS-CP036-powercast – Week 4 Section 1: Hyperparameter Optimization

Profile: `{PROFILE}`

### Key Questions Answered

**Q: Which hyperparameter tuning methods did you use, and why?**  
**A:** We used **lightweight grid search** tailored to runtime: a small, high-impact set for **XGBoost** (trees & learning rate) and 2 safe candidates for **SARIMAX** ((1,0,1) vs (1,1,1) with daily seasonality). This keeps dev runs **fast** while still exploring the parameters that move the needle.

**Q: What were the most important hyperparameters and how were ranges chosen?**  
**A:**  
- **XGBoost:** `n_estimators`, `max_depth`, `learning_rate` — they control **model capacity** and **how fast it learns**. Ranges are narrow by design for speed, widened in `preprod/final` if needed.  
- **SARIMAX:** difference order `d` and seasonal structure — governs **memory** and **daily cycle** fit.

**Q: How did you avoid data leakage or overfitting during tuning?**  
**A:** We tune on a **validation slice** only (train→validate split in time order). The **test set is untouched** until Section 2. We also **cap model sizes** and use **hourly resampling** for SARIMAX to keep complexity in check.

### Where to look
- Full tuning table: `{rel(tuning_csv, BASE_DIR)}`  
- Best-per-zone params: `{rel(best_json, BASE_DIR)}`

### Business Value (Executive View)
- **Faster learning loops** by focusing on parameters that matter.  
- **Safer decisions** (no leakage) → test numbers you can trust.  
- **Right-sized models** that balance accuracy and runtime.
"""
write_text(s1_md, report)

# ---- Consolidated upsert (Week 4) ----
def upsert_section(consolidated_path: Path, section_hdr_regex, new_block: str, after_hdr_regex=None):
    if not consolidated_path.exists():
        consolidated_path.write_text("\n".join([
            "# SDS-CP036-powercast – Week 4 (Business Report)",
            "",
            "## Table of Contents",
            "- [Section 1 – Hyperparameter Optimization](#sds-cp036-powercast-–-week-4-section-1-hyperparameter-optimization)",
            "- [Section 2 – Retraining & Validation](#sds-cp036-powercast-–-week-4-section-2-retraining--validation)",
            "- [Section 3 – Model Selection & Trade-offs](#sds-cp036-powercast-–-week-4-section-3-model-selection--trade-offs)",
            "- [Section 4 – Model Export & Deployment](#sds-cp036-powercast-–-week-4-section-4-model-export--deployment)",
            "- [Section 5 – Monitoring & Maintenance Planning](#sds-cp036-powercast-–-week-4-section-5-monitoring--maintenance-planning)",
            ""
        ]), encoding="utf-8")
    text = consolidated_path.read_text(encoding="utf-8")
    hdr_any = re.compile(r"(?m)^\#\s*SDS-CP036-powercast\s*[–-]\s*Week\s*4\s*Section\s*[1-5]\s*:\s*.*$")
    # remove existing
    m = re.search(section_hdr_regex, text, flags=re.M)
    if m:
        start = m.start()
        m2 = re.search(hdr_any, text[m.end():])
        end = (m.end()+m2.start()) if m2 else len(text)
        text = (text[:start] + text[end:]).strip()
    # insert after optional header
    if after_hdr_regex:
        m_after = re.search(after_hdr_regex, text, flags=re.M)
        if m_after:
            mnext = re.search(hdr_any, text[m_after.end():])
            end_after = (m_after.end()+mnext.start()) if mnext else len(text)
            text = text[:end_after].rstrip() + "\n\n" + new_block.strip() + "\n\n" + text[end_after:].lstrip()
        else:
            text = text.rstrip() + "\n\n" + new_block.strip() + "\n"
    else:
        text = text.rstrip() + "\n\n" + new_block.strip() + "\n"
    consolidated_path.write_text(text, encoding="utf-8")

upsert_section(
    CONSOLIDATED,
    r"^\#\s*SDS-CP036-powercast\s*[–-]\s*Week\s*4\s*Section\s*1\s*:\s*Hyperparameter\s*Optimization\s*$",
    s1_md.read_text(encoding="utf-8")
)
print("✅ Upserted Section 1 into consolidated:", rel(CONSOLIDATED, BASE_DIR))


[BOOT] BASE_DIR=/home/6376f5a9-d12b-4255-9426-c0091ad440a7/Powercast PROFILE=dev -> /home/6376f5a9-d12b-4255-9426-c0091ad440a7/Powercast/results/Wk04_Section1_dev
[FREQ] step=0 days 00:10:00 freq='10T' steps/day=144
[ZONE] Zone 1 Power Consumption
✅ Saved tuning results to: results/Wk04_Section1_dev/reports/tuning_results.csv
✅ Saved best params JSON to: results/Wk04_Section1_dev/reports/best_params.json
Wrote: /home/6376f5a9-d12b-4255-9426-c0091ad440a7/Powercast/results/Wk04_Section1_dev/reports/SDS-CP036-powercast_Wk04_Section1_Business_Report.md
✅ Upserted Section 1 into consolidated: SDS-CP036-powercast_Wk04_Report_Business.md
